# Deep Agent (딥 에이전트)

**Deep Agent**는 데이터 분석처럼 계획(planning), 코드 실행(code execution),
아티팩트(스크립트, 리포트, 시각화 등) 생성이 필요한 복잡한 작업을 처리하기 위해 설계된 에이전트입니다.

이 노트북에서는 Deep Agent를 사용하여:
1. CSV 파일을 분석하고
2. 커스텀 도구를 통해 결과를 공유하는

데이터 분석 에이전트를 구축합니다.

**참고**:
- [LangChain 공식 문서 - Deep Agents](https://docs.langchain.com/oss/python/deepagents)
- [LangChain 공식 문서 - Data Analysis](https://docs.langchain.com/oss/python/deepagents/data-analysis)
- [LangChain 공식 문서 - Backends](https://docs.langchain.com/oss/python/deepagents/backends)

**핵심 개념**:
- **Backend**: 샌드박스 환경에서 코드를 실행하는 백엔드 시스템
- **Custom Tools**: 외부 서비스(Slack, 이메일 등)와 연동하는 커스텀 도구
- **Checkpointer**: 멀티턴 대화를 지원하는 체크포인터

## Deep Agent 아키텍처 이해

Deep Agent는 일반 에이전트와 다음과 같은 차이가 있습니다:

### 일반 에이전트 vs Deep Agent

| 특성 | 일반 에이전트 | Deep Agent |
|------|-------------|------------|
| **코드 실행** | 도구를 통한 간접 실행 | 백엔드에서 직접 실행 |
| **파일 생성** | 제한적 | 스크립트, 리포트, 시각화 등 자유롭게 생성 |
| **샌드박스** | 없음 | 격리된 환경에서 안전하게 실행 |
| **계획 수립** | 단순 반복 | 복잡한 작업의 계획과 실행 |
| **아티팩트** | 텍스트 응답 위주 | 파일, 이미지, 코드 등 다양한 아티팩트 생성 |

### 실행 흐름

```
사용자 요청
    ↓
Deep Agent (계획 수립)
    ↓
Backend (코드 실행)
    ↓
아티팩트 생성 (스크립트, 차트, 리포트)
    ↓
커스텀 도구 (결과 공유)
    ↓
사용자에게 응답
```

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

## 1. 설치

Deep Agent를 사용하려면 `deepagents` 패키지를 설치해야 합니다.

```bash
pip install deepagents
```

선택적 의존성 (Slack 연동 시):
```bash
pip install slack-sdk
```

In [ ]:
# !pip install deepagents

## 2. Backend 설정

Deep Agent는 **Backend**를 통해 샌드박스 환경에서 코드를 실행합니다.
여러 백엔드 제공자를 사용할 수 있습니다:

| 백엔드 | 설명 | 보안 |
|--------|------|------|
| **Daytona** | 클라우드 샌드박스 | 높음 |
| **Modal** | 서버리스 샌드박스 | 높음 |
| **Runloop** | 클라우드 개발 환경 | 높음 |
| **LocalShell** | 로컬 셸 (개발/테스트용) | 낮음 (주의 필요) |

> **주의**: `LocalShellBackend`는 파일시스템과 셸에 대한 제한 없는 접근을 제공합니다.
> 개발/테스트 환경에서만 사용하세요. 프로덕션에서는 샌드박스 백엔드를 권장합니다.

> **Windows 참고**: `LocalShellBackend`의 내장 파일 도구(`read_file`, `ls`)는 Windows 절대 경로를
> 지원하지 않습니다. Windows에서는 에이전트가 `execute` 도구로 Python 명령을 실행하여
> 파일을 읽도록 프롬프트에서 안내하는 것이 좋습니다.

In [ ]:
# 작업 디렉토리 절대 경로
# 로컬 셸 백엔드 설정 (개발/테스트용)
# root_dir에 절대 경로를 지정하면 execute 명령이 이 디렉토리에서 실행됩니다
# 서브프로세스 출력은 부모 프로세스(주피터 커널)의 기본 인코딩으로 디코딩되므로,
# 자식 python의 출력 인코딩을 부모와 일치시킵니다 (한글 Windows: cp949).
# ':replace' — 인코딩할 수 없는 문자는 오류 대신 대체 문자로 처리
# 백엔드 동작 확인

In [2]:
from deepagents.backends import LocalShellBackend

# 작업 디렉토리 절대 경로
WORK_DIR = os.getcwd()

# 로컬 셸 백엔드 설정 (개발/테스트용)
# root_dir에 절대 경로를 지정하면 execute 명령이 이 디렉토리에서 실행됩니다
import locale

# 서브프로세스 출력은 부모 프로세스(주피터 커널)의 기본 인코딩으로 디코딩되므로,
# 자식 python의 출력 인코딩을 부모와 일치시킵니다 (한글 Windows: cp949).
# ':replace' — 인코딩할 수 없는 문자는 오류 대신 대체 문자로 처리
SUBPROCESS_ENC = locale.getpreferredencoding(False)

backend = LocalShellBackend(
    root_dir=WORK_DIR,
    env={
        "PATH": os.environ.get("PATH", "/usr/bin:/bin"),  # execute 도구가 python 명령을 찾는 데 필요
        "PYTHONIOENCODING": f"{SUBPROCESS_ENC}:replace",
    },
)

# 백엔드 동작 확인
result = backend.execute("echo ready")
print(result)
print(f"작업 디렉토리: {WORK_DIR}")

C:\Users\KOSA\anaconda3\envs\langchain\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


ExecuteResponse(output='ready\n', exit_code=0, truncated=False)
작업 디렉토리: C:\Users\KOSA\Desktop\Agentic_AI_GPT_MCP-main\04_Deep_Agent


In [5]:
# os.environ
os.environ['PATH']

'C:\\Users\\KOSA\\anaconda3\\envs\\langchain;C:\\Users\\KOSA\\anaconda3\\envs\\langchain\\Library\\mingw-w64\\bin;C:\\Users\\KOSA\\anaconda3\\envs\\langchain\\Library\\usr\\bin;C:\\Users\\KOSA\\anaconda3\\envs\\langchain\\Library\\bin;C:\\Users\\KOSA\\anaconda3\\envs\\langchain\\Scripts;C:\\Users\\KOSA\\anaconda3\\envs\\langchain\\bin;C:\\Users\\KOSA\\anaconda3\\condabin;C:\\Program Files\\Common Files\\Oracle\\Java\\javapath;C:\\WINDOWS\\system32;C:\\WINDOWS;C:\\WINDOWS\\System32\\Wbem;C:\\WINDOWS\\System32\\WindowsPowerShell\\v1.0;C:\\WINDOWS\\System32\\OpenSSH;C:\\Program Files\\Bandizip;C:\\Users\\kosa\\AppData\\Roaming\\Python\\Python313\\Scripts;C:\\Program Files\\Docker\\Docker\\resources\\bin;C:\\Program Files\\nodejs;C:\\Program Files\\dotnet;C:\\Program Files\\Git\\cmd;C:\\Users\\KOSA\\AppData\\Local\\Programs\\Python\\Launcher;C:\\Users\\KOSA\\AppData\\Local\\Microsoft\\WindowsApps;C:\\Users\\KOSA\\AppData\\Local\\Programs\\Microsoft VS Code\\bin;D:\\react-course\\tools\\jdk

## 3. 샘플 데이터 준비

분석할 CSV 데이터를 생성하고 백엔드에 업로드합니다.

In [8]:
WORK_DIR

'C:\\Users\\KOSA\\Desktop\\Agentic_AI_GPT_MCP-main\\04_Deep_Agent'

In [ ]:
# 샘플 판매 데이터 생성
# CSV 파일을 로컬 파일시스템에 직접 저장
# 파일 내용 확인

In [9]:
import csv
import io
# 샘플 판매 데이터 생성
data = [
    ["Date", "Product", "Units Sold", "Revenue"],
    ["2025-08-01", "사과", 10, 250],
    ["2025-08-02", "바나나", 5, 125],
    ["2025-08-03", "사과", 7, 175],
    ["2025-08-04", "체리", 3, 90],
    ["2025-08-05", "바나나", 8, 200],
    ["2025-08-06", "사과", 12, 300],
    ["2025-08-07", "체리", 6, 180],
    ["2025-08-08", "바나나", 9, 225],
    ["2025-08-09", "사과", 15, 375],
    ["2025-08-10", "체리", 4, 120],
]
# CSV 파일을 로컬 파일시스템에 직접 저장
data_dir = os.path.join(WORK_DIR, "data")
os.makedirs(data_dir, exist_ok=True)
csv_path = os.path.join(data_dir, "sales_data.csv")

with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerows(data)
print("데이터 저장 완료!")

print(f"파일 경로: {csv_path}")
print(f"행 수: {len(data) - 1}")  # 헤더 제외
print(f"컬럼: {data[0]}")
# 파일 내용 확인
import pandas as pd
df_check = pd.read_csv(csv_path)
df_check.head()

데이터 저장 완료!
파일 경로: C:\Users\KOSA\Desktop\Agentic_AI_GPT_MCP-main\04_Deep_Agent\data\sales_data.csv
행 수: 10
컬럼: ['Date', 'Product', 'Units Sold', 'Revenue']


,Date,Product,Units Sold,Revenue
0,2025-08-01,사과,10,250
1,2025-08-02,바나나,5,125
2,2025-08-03,사과,7,175
3,2025-08-04,체리,3,90
4,2025-08-05,바나나,8,200


## 4. 커스텀 도구 구현

Deep Agent에게 추가 기능을 제공하는 커스텀 도구를 만듭니다.
에이전트가 생성한 아티팩트(리포트, 시각화 등)를 외부로 공유할 수 있습니다.

여기서는 **로컬 파일 저장** 도구를 만들어 결과를 저장합니다.

In [ ]:
def save_report(text: str, file_path: str | None = None) -> str:
def download_artifact(file_path: str) -> str:
    # 백엔드에서 지정된 경로의 파일을 다운로드 (리스트로 전달)
    # 파일이 존재하면 성공 메시지와 파일 크기(bytes) 반환
    # 파일을 찾지 못한 경우 실패 메시지 반환

In [7]:
from langchain.tools import tool

@tool
def save_report(text: str, file_path: str | None = None) -> str:
    """분석 결과를 로컬 파일에 저장합니다.

    Args:
        text: 저장할 텍스트 내용 (분석 리포트)
        file_path: 저장할 파일 경로. 지정하지 않으면 기본 경로에 저장됩니다.
    """
    if not file_path:
        file_path = os.path.join(WORK_DIR, "data", "analysis_report.txt")

    dir_name = os.path.dirname(file_path)
    if dir_name:
        os.makedirs(dir_name, exist_ok=True)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)

    return f"리포트가 '{file_path}'에 저장되었습니다."

@tool
def download_artifact(file_path: str) -> str:
    """백엔드에서 생성된 아티팩트(이미지, 파일 등)를 다운로드합니다.
    Args:
        file_path: 다운로드할 파일의 경로
    """
    # 백엔드에서 지정된 경로의 파일을 다운로드 (리스트로 전달)
    files = backend.download_files([file_path])
    
    # 파일이 존재하면 성공 메시지와 파일 크기(bytes) 반환
    if files:
        return f"'{file_path}' 파일을 성공적으로 다운로드했습니다. ({len(files[0].content)} bytes)"
    
    # 파일을 찾지 못한 경우 실패 메시지 반환
    return f"'{file_path}' 파일을 찾을 수 없습니다."

print("커스텀 도구 생성 완료:")
print(f"  - save_report: {save_report.description[:50]}...")
print(f"  - download_artifact: {download_artifact.description[:50]}...")

커스텀 도구 생성 완료:
  - save_report: 분석 결과를 로컬 파일에 저장합니다.

    Args:
        text: 저장할 ...
  - download_artifact: 백엔드에서 생성된 아티팩트(이미지, 파일 등)를 다운로드합니다.
    Args:
    ...


## 5. Deep Agent 생성 및 실행

`create_deep_agent()` 팩토리 함수로 에이전트를 생성합니다.

### 내장 도구 (기본 제공)

create_deep_agent는 별도 등록 없이 다음 도구들을 자동으로 포함합니다.
아래 실행 출력에서 에이전트가 이 도구들을 호출하는 것을 볼 수 있습니다.

| 내장 도구 | 역할 |
|---|---|
| `write_todos` | 작업 계획 수립/추적 (복잡한 작업을 단계로 분해) |
| `ls`, `read_file`, `write_file`, `edit_file` | 백엔드 파일시스템 탐색/읽기/쓰기/수정 |
| `task` | 서브에이전트 호출 (독립 컨텍스트로 하위 작업 위임) |
| `execute` | 셸 명령/코드 실행 (Sandbox 백엔드 연결 시) |

여기에 `tools` 파라미터로 **확장 도구**(웹 검색, MCP 어댑터, 커스텀 도구 등)를 추가할 수 있습니다.

### 주요 파라미터

- **model**: 사용할 LLM 모델
- **tools**: 커스텀 도구 목록
- **backend**: 코드 실행 백엔드 (기본: 인메모리 상태 기반, 교체 가능)
- **system_prompt**: 에이전트 역할/지침
- **checkpointer**: 멀티턴 대화 지원용 체크포인터
- **subagents / skills / memory / interrupt_on**: 고급 기능 (→ [401_Deep_Agent_Advanced.ipynb](401_Deep_Agent_Advanced.ipynb))


In [ ]:
# 체크포인터 생성 - 멀티턴 대화의 상태를 메모리에 저장
# Deep Agent 생성
# 고유한 스레드 ID 생성 - 대화 세션을 식별하는 데 사용

In [10]:
import uuid
from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent

# 체크포인터 생성 - 멀티턴 대화의 상태를 메모리에 저장
checkpointer = InMemorySaver()

# Deep Agent 생성
agent = create_deep_agent(
    model="openai:gpt-5-mini",       # 사용할 LLM 모델
    tools=[save_report, download_artifact],  # 커스텀 도구 등록
    backend=backend,                  # 코드 실행 백엔드 (LocalShell)
    system_prompt=(                   # 에이전트 역할/지침
        "당신은 데이터 분석 전문가입니다. "
        "계획을 세운 뒤 코드를 실행해 데이터를 분석하고, 결과를 리포트로 정리하세요."
    ),
    checkpointer=checkpointer,        # 대화 상태 관리용 체크포인터
)

# 고유한 스레드 ID 생성 - 대화 세션을 식별하는 데 사용
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

print(f"Deep Agent 생성 완료 (thread_id: {thread_id})")

Deep Agent 생성 완료 (thread_id: 0790cfcc-79e0-4f69-a90d-46fefe35e7ba)


### 에이전트에게 분석 요청

자연어로 데이터 분석을 요청하면, 에이전트가 자동으로:
1. 데이터를 읽고 탐색
2. 분석 코드를 작성하고 실행
3. 커스텀 도구를 사용하여 결과를 공유

In [ ]:
# NOTE: LocalShellBackend의 read_file/ls 도구는 Windows 경로를 지원하지 않으므로,
# 에이전트가 execute 도구로 Python/pandas를 사용해 파일을 읽도록 안내합니다.
# Windows 백슬래시(\)를 슬래시(/)로 변환 - Agent가 경로를 올바르게 인식하도록
# Agent에게 전달할 사용자 메시지 구성
# - CSV 파일 경로와 분석 지시사항을 포함
# - read_file/ls 대신 execute 도구 사용을 명시 (Windows 호환성)
# Agent 실행 - 스트리밍 모드로 각 단계의 업데이트를 실시간 출력
    # 각 스텝에서 메시지 추출 후 출력

In [11]:
# NOTE: LocalShellBackend의 read_file/ls 도구는 Windows 경로를 지원하지 않으므로,
# 에이전트가 execute 도구로 Python/pandas를 사용해 파일을 읽도록 안내합니다.

# Windows 백슬래시(\)를 슬래시(/)로 변환 - Agent가 경로를 올바르게 인식하도록
csv_path_for_agent = csv_path.replace("\\", "/")

# Agent에게 전달할 사용자 메시지 구성
# - CSV 파일 경로와 분석 지시사항을 포함
# - read_file/ls 대신 execute 도구 사용을 명시 (Windows 호환성)
input_message = {
    "role": "user",
    "content": (
        f"'{csv_path_for_agent}' 경로에 있는 CSV 파일을 분석해주세요. "
        "반드시 execute 도구를 사용하여 Python(pandas)으로 데이터를 읽고 분석하세요. "
        "데이터가 한달치 이므로 일별 매출 추세를 분석하세요."
        "레포트는 한국어로 작성하세요."
        "read_file이나 ls 도구는 사용하지 마세요. "
        "분석이 끝나면 save_report 도구를 사용하되, "
        "file_path 인자 없이 호출하여 기본 경로에 리포트를 저장해주세요."
    ),
}

# Agent 실행 - 스트리밍 모드로 각 단계의 업데이트를 실시간 출력
for step in agent.stream(
    {"messages": [input_message]},  # 사용자 메시지를 입력으로 전달
    config,                         # thread_id가 포함된 설정
    # stream_mode="values",          # 매 스텝 후 전체 상태 스냅샷
    stream_mode="updates",          # 각 노드의 업데이트만 스트리밍
    # stream_mode="messages",          # LLM 토큰 단위 — 실시간 타이핑 효과
):
    # 각 스텝에서 메시지 추출 후 출력
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()  # LangChain 메시지 포맷으로 보기 좋게 출력

================================== Ai Message ==================================

[{'id': 'rs_0187414694d87284006a4f3d695d2481998f257860338cf8e1', 'summary': [], 'type': 'reasoning', 'content': []}, {'arguments': '{"todos":[{"content":"CSV 파일을 Python(pandas)으로 읽어 초기 확인하기","status":"in_progress"},{"content":"일별 매출 합계 계산 및 시계열 데이터 생성","status":"pending"},{"content":"기초 통계(총합, 평균, 최고/최저일) 및 추세(선형 회귀) 분석","status":"pending"},{"content":"시각화(일별 매출 선 그래프) 생성 및 이미지 저장","status":"pending"},{"content":"분석 결과를 한국어 보고서로 작성하고 저장(save_report 호출)","status":"pending"}]}', 'call_id': 'call_fsouYPQbf0rgzJSUIMIQjUaw', 'name': 'write_todos', 'type': 'function_call', 'id': 'fc_0187414694d87284006a4f3d7755a88199996daf781d42cb7c', 'status': 'completed'}]
Tool Calls:
  write_todos (call_fsouYPQbf0rgzJSUIMIQjUaw)
 Call ID: call_fsouYPQbf0rgzJSUIMIQjUaw
  Args:
    todos: [{'content': 'CSV 파일을 Python(pandas)으로 읽어 초기 확인하기', 'status': 'in_progress'}, {'content': '일별 매출 합계 계산 및 시계열 데이터 생성', 'status': 'pending'}, 

Exception in thread Thread-8 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\KOSA\anaconda3\envs\langchain\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\KOSA\anaconda3\envs\langchain\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\KOSA\anaconda3\envs\langchain\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
UnicodeDecodeError: 'cp949' codec can't decode byte 0x80 in position 2: illegal multibyte sequence


================================== Ai Message ==================================

[{'id': 'rs_0187414694d87284006a4f3d8ab0988199b0b19e1d2f73c3a2', 'summary': [], 'type': 'reasoning', 'content': []}, {'arguments': '{"command":"powershell -NoProfile -Command \\"$script = @\'\\nimport pandas as pd\\nimport numpy as np\\nimport matplotlib\\nmatplotlib.use(\'Agg\')\\nimport matplotlib.pyplot as plt\\nimport sys\\n\\npath = r\'C:/Users/KOSA/Desktop/Agentic_AI_GPT_MCP-main/04_Deep_Agent/data/sales_data.csv\'\\nprint(\'=== READING CSV ===\')\\ntry:\\n    df = pd.read_csv(path)\\nexcept Exception as e:\\n    print(\'ERROR_READING_FILE:\', e)\\n    sys.exit(1)\\n\\nprint(\'Shape:\', df.shape)\\nprint(\'Columns:\', list(df.columns))\\nprint(\'\\\\n--- DTYPE INFO ---\')\\nprint(df.dtypes)\\nprint(\'\\\\n--- HEAD (first 10 rows) ---\')\\nprint(df.head(10).to_string(index=False))\\n\\n# Identify date column\\ndate_col = None\\nfor c in df.columns:\\n    if \'date\' in c.lower() or \'day\' in c.lower

## 6. 멀티턴 대화

체크포인터 덕분에 동일한 `thread_id`로 후속 질문을 할 수 있습니다.
에이전트는 이전 대화 맥락을 기억합니다.

In [ ]:
# 후속 질문 메시지 구성
# - 이전 대화 컨텍스트(체크포인터)를 활용한 멀티턴 대화
# 후속 질문 실행 - 동일한 config(thread_id)로 대화 이어가기
    # 각 스텝에서 메시지 추출 후 출력

In [12]:
# 후속 질문 메시지 구성
# - 이전 대화 컨텍스트(체크포인터)를 활용한 멀티턴 대화
followup_message = {
    "role": "user",
    "content": (
        "가장 잘 팔린 제품은 무엇인가요? 그리고 일별 매출 추세를 분석해주세요. "
        "execute 도구로 Python 코드를 실행하여 분석하세요."
        "레포트는 한국어로 작성하세요."
    ),
}

# 후속 질문 실행 - 동일한 config(thread_id)로 대화 이어가기
for step in agent.stream(
    {"messages": [followup_message]},  # 후속 질문 전달
    config,                            # 동일한 thread_id로 대화 연속성 유지
    stream_mode="updates",
):
    # 각 스텝에서 메시지 추출 후 출력
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_0187414694d87284006a4f3f5bd1048199848019349b4ff518', 'summary': [], 'type': 'reasoning', 'content': []}, {'arguments': '{"todos":[{"content":"CSV 파일을 Python(pandas)으로 읽어 초기 확인하기","status":"completed"},{"content":"일별 매출 합계 계산 및 시계열 데이터 생성","status":"in_progress"},{"content":"기초 통계(총합, 평균, 최고/최저일) 및 추세(선형 회귀) 분석","status":"pending"},{"content":"시각화(일별 매출 선 그래프) 및 상위 제품 일별 매출 그래프 생성","status":"pending"},{"content":"분석 결과를 한국어 보고서로 작성하고 저장(save_report 호출)","status":"pending"}]}', 'call_id': 'call_IbmaFpdG3PMImH4t1hOu3VaQ', 'name': 'write_todos', 'type': 'function_call', 'id': 'fc_0187414694d87284006a4f3f68ff9c8199960449d0f07c76db', 'status': 'completed'}]
Tool Calls:
  write_todos (call_IbmaFpdG3PMImH4t1hOu3VaQ)
 Call ID: call_IbmaFpdG3PMImH4t1hOu3VaQ
  Args:
    todos: [{'content': 'CSV 파일을 Python(pandas)으로 읽어 초기 확인하기', 'status': 'completed'}, {'content': '일별 매출 합계 계산 및 시계열 데이터 생성', 'status': 'in

## 7. 아티팩트 확인

에이전트가 생성한 파일(시각화 차트, 리포트 등)을 확인합니다.

In [ ]:
# === 저장된 리포트 파일 확인 ===
    # 리포트 파일이 존재하면 내용 출력

In [13]:
# === 저장된 리포트 파일 확인 ===
default_report_path = os.path.join(WORK_DIR, "data", "analysis_report.txt")

if os.path.exists(default_report_path):
    # 리포트 파일이 존재하면 내용 출력
    with open(default_report_path, "r", encoding="utf-8") as f:
        print("=== 저장된 분석 리포트 ===")
        print(f.read())
else:
    print(f"리포트 파일이 아직 생성되지 않았습니다: {default_report_path}")

=== 저장된 분석 리포트 ===
분석 개요 (요청받은 작업)
- 파일 경로: C:/Users/KOSA/Desktop/Agentic_AI_GPT_MCP-main/04_Deep_Agent/data/sales_data.csv
- 요구사항: execute 도구를 사용하여 Python(pandas)로 CSV를 읽고, 한 달치 데이터를 일별 매출로 집계해 추세를 분석. 결과는 한국어 레포트로 저장.

실행한 작업(절차)
1) Python(pandas) 환경에서 CSV 파일을 읽도록 명령을 실행했습니다 (execute 도구 사용).
2) 날짜 컬럼을 자동으로 탐색하여 datetime으로 변환했습니다. 컬럼명이 'date' 또는 'day'를 포함하는 컬럼을 우선으로 선택하고, 없으면 첫 번째 컬럼을 날짜로 시도했습니다.
3) 매출(판매) 컬럼을 자동으로 탐색했습니다. 우선적으로 'sales','revenue','amount','total','price' 등 후보명을 검색하고, 없을 경우 'price'와 'quantity/qty'로부터 매출(revenue=price*qty)을 계산하도록 시도했습니다. 그래도 없으면 숫자형 컬럼 중 ID·code 계열을 제외한 첫 번째 컬럼을 대체 매출 열로 사용했습니다.
4) 날짜별로 합계를 계산하여 일별 시계열(daily sales)을 생성했습니다. 데이터에 빠진 날짜는(범위 내) 0으로 채웠습니다.
5) 통계값(총합, 평균, 중앙값, 표준편차), 최고/최저 일자와 값, 선형 회귀를 이용한 추세(일 단위 인덱스에 대한 기울기)를 계산했습니다.
6) 일별 매출 선 그래프를 생성하여 'daily_sales.png'로 저장하도록 시도했습니다.

실행 결과 및 현황
- 분석을 수행하기 위해 Python 코드를 execute 도구로 여러 번 실행했습니다. (pandas 임포트 확인 및 분석 스크립트 실행 포함)
- 실행 환경에서 matplotlib의 설정 디렉터리(MPLCONFIGDIR)를 지정하여 플롯 저장을 시도했습니다.
- 다만, 현재 분석 

## 주요 포인트 정리

1. **Deep Agent**: 계획, 코드 실행, 아티팩트 생성이 가능한 고급 에이전트
2. **Backend**: 샌드박스 환경에서 안전하게 코드를 실행 (Daytona, Modal, Runloop, LocalShell)
3. **Custom Tools**: 외부 서비스와 연동하는 도구를 만들어 결과를 공유
4. **Checkpointer**: 멀티턴 대화를 지원하여 이전 맥락을 유지
5. **Artifacts**: 스크립트, 리포트, 시각화 등 다양한 아티팩트를 생성하고 관리

**다음 단계**:
- [401_Deep_Agent_Advanced.ipynb](401_Deep_Agent_Advanced.ipynb): Human-in-the-Loop, Skills, Memory 고급 기능 실습
- [Deep Agents Customization](https://docs.langchain.com/oss/python/deepagents/customization): 모델, 도구, 프롬프트, 계획 전략 커스터마이징
- [Deep Agents Sandboxes](https://docs.langchain.com/oss/python/deepagents/sandboxes): 샌드박스 제공자별 설정 가이드
- [Deep Agents Skills](https://docs.langchain.com/oss/python/deepagents/skills): 재사용 가능한 스킬 구현

## 실습문제: 나만의 데이터 분석 에이전트

본문의 흐름(데이터 준비 → 커스텀 도구 → Deep Agent 생성 → 분석 요청 → 멀티턴)을 새로운 데이터에 그대로 적용해 보세요.

1. **데이터 준비**: 도서 판매 데이터 `data/book_sales.csv`를 만드세요.
   - 컬럼: Date, Genre, Units Sold, Revenue — 장르 3종(소설/에세이/과학) 8행 정도면 충분합니다.
2. **커스텀 도구**: 요약 텍스트를 `data/book_summary.txt`에 저장하는 `save_summary` 도구를 만드세요. (본문 `save_report`와 같은 구조)
3. **에이전트 생성 + 분석**: `create_deep_agent`(model, tools, backend, system_prompt, checkpointer)로 에이전트를 만들고 **장르별 매출 분석**을 요청하세요.
   - 본문처럼 execute 도구로 pandas를 사용하도록 지시하고, 경로는 슬래시(/)로 변환해 전달하세요.
4. **멀티턴**: 같은 thread_id로 후속 질문(예: "가장 매출이 높은 장르는?")을 던져 이전 분석 맥락이 유지되는지 확인하세요.


# 실습 문제 모범답안 - 400. Deep Agent

`400_Deep_Agent` 노트북 실습 문제(나만의 데이터 분석 에이전트)에 대한 모범답안 예시입니다.

### 구현 요약

본문의 흐름을 도서 판매 데이터에 그대로 적용합니다.

1. 도서 판매 데이터 `data/book_sales.csv` 생성
2. 요약을 저장하는 `save_summary` 커스텀 도구
3. `create_deep_agent`로 에이전트 생성 후 장르별 매출 분석 요청
4. 같은 thread_id로 후속 질문 (멀티턴)

In [14]:
import csv
from deepagents.backends import LocalShellBackend

WORK_DIR = os.getcwd()

import locale

# 서브프로세스 출력은 부모 프로세스(주피터 커널)의 기본 인코딩으로 디코딩되므로,
# 자식 python의 출력 인코딩을 부모와 일치시킵니다 (한글 Windows: cp949).
# ':replace' — 인코딩할 수 없는 문자는 오류 대신 대체 문자로 처리
SUBPROCESS_ENC = locale.getpreferredencoding(False)

backend = LocalShellBackend(
    root_dir=WORK_DIR,
    env={
        "PATH": os.environ.get("PATH", "/usr/bin:/bin"),  # execute 도구가 python 명령을 찾는 데 필요
        "PYTHONIOENCODING": f"{SUBPROCESS_ENC}:replace",
    },
)

# 도서 판매 데이터 생성
data = [
    ["Date", "Genre", "Units Sold", "Revenue"],
    ["2025-09-01", "소설", 12, 180000],
    ["2025-09-02", "에세이", 8, 96000],
    ["2025-09-03", "과학", 5, 90000],
    ["2025-09-04", "소설", 15, 225000],
    ["2025-09-05", "에세이", 6, 72000],
    ["2025-09-06", "과학", 9, 162000],
    ["2025-09-07", "소설", 10, 150000],
    ["2025-09-08", "에세이", 11, 132000],
]

data_dir = os.path.join(WORK_DIR, "data")
os.makedirs(data_dir, exist_ok=True)
csv_path = os.path.join(data_dir, "book_sales.csv")

with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerows(data)

print(f"데이터 저장 완료: {csv_path} ({len(data) - 1}행)")

데이터 저장 완료: C:\Users\KOSA\Desktop\Agentic_AI_GPT_MCP-main\04_Deep_Agent\data\book_sales.csv (8행)


## 2. 커스텀 도구: save_summary

본문의 `save_report`와 같은 구조입니다.

In [15]:
from langchain.tools import tool

@tool
def save_summary(text: str) -> str:
    """분석 요약을 data/book_summary.txt 파일에 저장합니다.

    Args:
        text: 저장할 요약 텍스트
    """
    file_path = os.path.join(WORK_DIR, "data", "book_summary.txt")
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)
    return f"요약이 '{file_path}'에 저장되었습니다."

## 3. Deep Agent 생성 및 장르별 매출 분석 요청

본문과 같이 Windows 호환을 위해 execute 도구로 pandas를 사용하도록 지시하고,
경로의 백슬래시는 슬래시로 변환해 전달합니다.

In [16]:
import uuid
from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="openai:gpt-5-mini",
    tools=[save_summary],
    backend=backend,
    system_prompt=(
        "당신은 데이터 분석 전문가입니다. "
        "계획을 세운 뒤 코드를 실행해 데이터를 분석하고, 결과를 한국어로 정리하세요."
    ),
    checkpointer=InMemorySaver(),
)

thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

csv_path_for_agent = csv_path.replace("\\", "/")

input_message = {
    "role": "user",
    "content": (
        f"'{csv_path_for_agent}' 경로의 CSV 파일을 분석해주세요. "
        "반드시 execute 도구를 사용하여 Python(pandas)으로 데이터를 읽고, "
        "장르별 총 매출과 판매량을 분석하세요. "
        "Windows cmd 환경이므로: python3가 아닌 python 명령을 사용하고, "
        "heredoc(<<)이나 여러 줄 명령은 동작하지 않으니 "
        "세미콜론으로 이어진 한 줄짜리 python -c 명령만 사용하세요. "
        "read_file이나 ls 도구는 사용하지 마세요. "
        "분석이 끝나면 save_summary 도구로 한국어 요약을 저장해주세요."
    ),
}

for step in agent.stream(
    {"messages": [input_message]},
    config,
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_0e9fb59224e429ef006a4f40c22a48819a83fe94dc4b9b25f2', 'summary': [], 'type': 'reasoning', 'content': []}, {'arguments': '{"command":"python -c \\"import pandas as pd,sys,json;path=r\'C:/Users/KOSA/Desktop/Agentic_AI_GPT_MCP-main/04_Deep_Agent/data/book_sales.csv\';df=pd.read_csv(path);cols=list(df.columns);lc={c.lower():c for c in cols};find=lambda cands: next((lc[x.lower()] for x in cands if x.lower() in lc),None);genre=find([\'genre\',\'Genre\',\'category\',\'Category\',\'장르\',\'type\']);qty=find([\'quantity\',\'Quantity\',\'units\',\'Units\',\'sales_count\',\'count\',\'판매량\']);rev=find([\'revenue\',\'Revenue\',\'sales\',\'Sales\',\'amount\',\'Amount\',\'매출\',\'total\',\'Total\']);price=find([\'price\',\'Price\',\'unit_price\',\'Unit_Price\']);if genre is None: print(json.dumps({\'error\':\'GENRE_NOT_FOUND\',\'columns\':cols},ensure_ascii=False)); sys.exit(0);if rev is None and price is not N

## 4. 멀티턴 후속 질문 + 결과 확인

같은 `thread_id`로 후속 질문을 던지면, 에이전트가 이전 분석 결과를 기억한 채 답변합니다.

In [18]:
followup_message = {
    "role": "user",
    "content": "방금 분석에서 가장 매출이 높은 장르는 무엇이었나요? 한 문장으로 답해주세요.",
}

for step in agent.stream(
    {"messages": [followup_message]},
    config,  # 동일한 thread_id
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

================================== Ai Message ==================================

[{'id': 'rs_0e9fb59224e429ef006a4f415c9330819ab3aadc2af3435de5', 'summary': [], 'type': 'reasoning', 'content': []}, {'type': 'text', 'text': '가장 매출이 높은 장르는 소설(매출 555,000원)입니다.', 'annotations': [], 'id': 'msg_0e9fb59224e429ef006a4f415cecb4819a993c099251617556'}]


In [19]:
# 저장된 요약 파일 확인
summary_path = os.path.join(WORK_DIR, "data", "book_summary.txt")

if os.path.exists(summary_path):
    with open(summary_path, "r", encoding="utf-8") as f:
        print("=== 저장된 분석 요약 ===")
        print(f.read())
else:
    print(f"요약 파일이 아직 생성되지 않았습니다: {summary_path}")

=== 저장된 분석 요약 ===
데이터 경로: C:/Users/KOSA/Desktop/Agentic_AI_GPT_MCP-main/04_Deep_Agent/data/book_sales.csv

분석 목표: 장르별 총 매출(Revenue)과 판매량(Units Sold) 집계

분석 방법: Python(pandas)로 CSV를 읽고 'Genre'로 그룹화하여 'Units Sold'와 'Revenue'의 합계를 계산함.

결과 요약:
- 과학: 판매량 14권, 매출 252,000원
- 소설: 판매량 37권, 매출 555,000원
- 에세이: 판매량 25권, 매출 300,000원

전체 합계: 판매량 76권, 총매출 1,107,000원

비고: 사용된 칼럼명은 CSV에 'Genre', 'Units Sold', 'Revenue'로 존재하였음.


## 확인 포인트

- 스트리밍 출력에서 내장 도구(`write_todos`, `execute`)와 커스텀 도구(`save_summary`)가 호출되는 순서를 관찰하세요.
- 후속 질문에서 에이전트가 CSV를 다시 읽지 않고 **이전 분석 맥락**으로 답하면 멀티턴(체크포인터)이 동작하는 것입니다.
- `data/book_summary.txt` 파일이 생성되었으면 커스텀 도구 연동까지 완료된 것입니다.